In [1]:
import os
from dotenv import load_dotenv
from datasets import Dataset
from pandas import DataFrame

# Updated Ragas metric imports
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas import evaluate

# LangChain OpenAI integrations
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [2]:
# 1. Load the environment variables from your .env file
load_dotenv()

# Verify the key is loaded (don't print the actual key in output!)
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

# 2. Initialize the OpenAI Models natively
judge_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0  # 0 is strictly recommended for evaluation/judge tasks
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print(f"✅ Judge LLM configured as: {judge_model.model_name}")
print(f"✅ Embeddings configured as: {embeddings.model}")

✅ Judge LLM configured as: gpt-4o-mini
✅ Embeddings configured as: text-embedding-3-small


In [13]:
data = {
    'question': [
        'What is Kubernetes?',
        'What is Docker?'
    ],
    'answer': [
        'Kubernetes is an open-source system for automating deployment, scaling, and management of containerized applications.',
        # 'Docker is a platform designed to help developers build, share, and run modern applications.'
        'A Docker is a device that helps ship dock at the port.'
    ],
    'contexts': [
        ['Kubernetes is an open-source container orchestration system.', 'Docker is used to create and run containers either on bare-metal or cloud run virtualized infra on top of hypervisors.'],
        ['Kubernetes is an open-source container orchestration system.', 'Docker is used to create and run containers either on bare-metal or cloud run virtualized infra on top of hypervisors.']
    ],
    'reference': [
        'Kubernetes is an open-source system for managing containerized applications.',
        'Docker is a platform for building and running containers.'
    ]
}

dataset = Dataset.from_dict(data)
display(DataFrame(dataset))
display(DataFrame(dataset)[['contexts']][0:1].values[0][0][1])

,question,answer,contexts,reference
0,What is Kubernetes?,Kubernetes is an open-source system for automa...,[Kubernetes is an open-source container orches...,Kubernetes is an open-source system for managi...
1,What is Docker?,A Docker is a device that helps ship dock at t...,[Kubernetes is an open-source container orches...,Docker is a platform for building and running ...


'Docker is used to create and run containers either on bare-metal or cloud run virtualized infra on top of hypervisors.'

In [4]:
print("Evaluating model outputs concurrently via OpenAI...")

# Execute Ragas using the cloud models
score = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=judge_model,
    embeddings=embeddings,
    raise_exceptions=False # Prevents the entire run from failing if one evaluation errors out
)


# Output as a clean Pandas DataFrame for analysis
df_results = score.to_pandas()
display(DataFrame(df_results))

Evaluating model outputs concurrently via OpenAI...


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000018F21862E80> is already entered


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is Kubernetes?,[Kubernetes is an open-source container orches...,Kubernetes is an open-source system for automa...,Kubernetes is an open-source system for managi...,0.25,0.867267,1.0,1.0
1,What is Docker?,[Kubernetes is an open-source container orches...,A Docker is a device that helps ship dock at t...,Docker is a platform for building and running ...,0.00,0.962551,0.5,1.0
